In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# ResNet-50 — GPU Benchmark on CIFAR-100

**Enhancements over 7th-semester baseline:**

| # | Enhancement | Reviewer concern addressed |
|---|-------------|----------------------------|
| 1 | CIFAR-100 (100 classes) | *Narrow scope* |
| 2 | Cross-domain generalization (STL-10) | *Real-world domain shifts* |
| 3 | Energy efficiency (CodeCarbon) | *Energy metrics missing* |
| 4 | McNemar's statistical test | *No statistical validation* |
| 5 | ONNX INT-8 quantisation + CPU latency | *Deployability* |
| 6 | ECE / temperature scaling | *Calibration* |
| 7 | Corruption robustness (5 types) | *Robustness* |

**Hardware:** Kaggle GPU (CUDA)  
**Model:** `resnet50`  
**Dataset:** CIFAR-100 → STL-10 cross-domain

In [2]:
# ──────────────── Install & Import ────────────────
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!pip install -q onnx onnxruntime-gpu onnxsim psutil codecarbon \
              opencv-python-headless scipy thop

import os, math, time, io, copy, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import onnx, onnxruntime as ort, psutil
from onnxruntime.quantization import quantize_dynamic, QuantType
from scipy.stats import chi2
warnings.filterwarnings('ignore')

print(f'PyTorch  : {torch.__version__}')
print(f'ORT      : {ort.__version__}')
if torch.cuda.is_available():
    print(f'Device   : cuda  ({torch.cuda.get_device_name()})')
else:
    print('Device   : cpu')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 20.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.6/252.6 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.6/359.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-genai 1.57.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
a2a-sdk 0.3.22 requires httpx>=0.28.1,

In [3]:
# ──────────────── Configuration ────────────────
MODEL_NAME   = 'resnet50'
IMG_SIZE     = 224
BATCH        = 64
EPOCHS       = 15
LR           = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
WARMUP_EP    = 3
NUM_WORKERS  = 2
SEED         = 42
NUM_CLASSES  = 100          # CIFAR-100
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

RESULTS_DIR  = './results'
PRED_DIR     = './predictions'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Model        : {MODEL_NAME}')
print(f'Dataset      : CIFAR-100 ({NUM_CLASSES} classes)')
print(f'Batch / Epoch: {BATCH} / {EPOCHS}')
print(f'CPU threads  : {torch.get_num_threads()}')

Model        : resnet50
Dataset      : CIFAR-100 (100 classes)
Batch / Epoch: 64 / 15
CPU threads  : 2


In [4]:
# ──────────────── Data Loading ────────────────
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomCrop(IMG_SIZE, padding=IMG_SIZE // 8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

root = './data'

# ─── CIFAR-100 ───
train_full_split = datasets.CIFAR100(root, train=True, download=True,
                                     transform=transforms.ToTensor())
g = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(
    range(len(train_full_split)), [45000, 5000], generator=g)

train_full_aug   = datasets.CIFAR100(root, train=True, download=False,
                                     transform=train_tf)
train_full_plain = datasets.CIFAR100(root, train=True, download=False,
                                     transform=test_tf)

train_ds = Subset(train_full_aug,   train_idx.indices)
val_ds   = Subset(train_full_plain, val_idx.indices)
test_ds  = datasets.CIFAR100(root, train=False, download=True,
                             transform=test_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# ─── STL-10 (cross-domain generalization) ───
stl10_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
stl10_train = datasets.STL10(root, split='train', download=True,
                             transform=stl10_tf)
stl10_test  = datasets.STL10(root, split='test',  download=True,
                             transform=stl10_tf)

stl10_train_loader = DataLoader(stl10_train, batch_size=BATCH,
                                shuffle=True,  num_workers=NUM_WORKERS,
                                pin_memory=True)
stl10_test_loader  = DataLoader(stl10_test,  batch_size=BATCH,
                                shuffle=False, num_workers=NUM_WORKERS,
                                pin_memory=True)

print(f'CIFAR-100  ─ Train: {len(train_ds)}, Val: {len(val_ds)}, '
      f'Test: {len(test_ds)}')
print(f'STL-10     ─ Train: {len(stl10_train)}, Test: {len(stl10_test)}')

100%|██████████| 169M/169M [00:13<00:00, 12.7MB/s]
100%|██████████| 2.64G/2.64G [03:41<00:00, 11.9MB/s]


CIFAR-100  ─ Train: 45000, Val: 5000, Test: 10000
STL-10     ─ Train: 5000, Test: 8000


In [5]:
# ──────────────── Model & Training Utilities ────────────────
def get_model(num_classes=NUM_CLASSES):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m


def make_warmup_cosine(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses, y_true, y_pred = [], [], []
    scaler = torch.amp.GradScaler(enabled=(DEVICE == 'cuda'))

    if is_train:
        for X, Y in loader:
            X = X.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
                logits = model(X)
                loss = criterion(logits, Y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())
            y_true += Y.cpu().tolist()
            y_pred += logits.detach().argmax(1).cpu().tolist()
    else:
        with torch.no_grad():
            for X, Y in loader:
                X = X.to(DEVICE, non_blocking=True)
                Y = Y.to(DEVICE, non_blocking=True)
                logits = model(X)
                loss = criterion(logits, Y)
                losses.append(loss.item())
                y_true += Y.cpu().tolist()
                y_pred += logits.detach().argmax(1).cpu().tolist()

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    pre = precision_score(y_true, y_pred, average='macro')
    rec = recall_score(y_true, y_pred, average='macro')
    return float(np.mean(losses)), dict(acc=acc, f1=f1,
                                        precision=pre, recall=rec)

In [6]:
# ──────────────── Training ────────────────
print(f'\n{"="*60}')
print(f'  Training {MODEL_NAME} on CIFAR-100  (CPU)')
print(f'{"="*60}\n')

model = get_model(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer = optim.AdamW(model.parameters(), lr=LR,
                        weight_decay=WEIGHT_DECAY)
scheduler = make_warmup_cosine(optimizer, WARMUP_EP, EPOCHS)

best_acc, best_state = 0.0, None
history = []

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl = run_epoch(model, val_loader,   criterion, None)
    scheduler.step()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    elapsed = time.time() - t0

    history.append(dict(epoch=ep, train_loss=tr_loss, val_loss=vl_loss,
                        train_acc=tr['acc'], val_acc=vl['acc'],
                        val_f1=vl['f1'], time_s=elapsed))

    if vl['acc'] > best_acc:
        best_acc = vl['acc']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f'[{MODEL_NAME}] ep {ep:02d}/{EPOCHS} │ '
          f'val_acc={vl["acc"]:.4f}  f1={vl["f1"]:.4f} │ '
          f'{elapsed:.1f}s')

# restore best & evaluate
model.load_state_dict(best_state)
torch.save(model.state_dict(),
           f'{RESULTS_DIR}/{MODEL_NAME}_cifar100.pt')
_, test_metrics = run_epoch(model, test_loader, criterion, None)
print(f'\n[{MODEL_NAME}] TEST: {test_metrics}')


  Training resnet50 on CIFAR-100  (CPU)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 228MB/s]


[resnet50] ep 01/15 │ val_acc=0.7314  f1=0.7235 │ 202.1s
[resnet50] ep 02/15 │ val_acc=0.6826  f1=0.6794 │ 200.0s
[resnet50] ep 03/15 │ val_acc=0.6782  f1=0.6724 │ 199.3s
[resnet50] ep 04/15 │ val_acc=0.7180  f1=0.7130 │ 200.1s
[resnet50] ep 05/15 │ val_acc=0.7366  f1=0.7311 │ 199.6s
[resnet50] ep 06/15 │ val_acc=0.7474  f1=0.7429 │ 199.7s
[resnet50] ep 07/15 │ val_acc=0.7692  f1=0.7660 │ 199.5s
[resnet50] ep 08/15 │ val_acc=0.7798  f1=0.7763 │ 199.6s
[resnet50] ep 09/15 │ val_acc=0.7998  f1=0.7952 │ 201.2s
[resnet50] ep 10/15 │ val_acc=0.8036  f1=0.7997 │ 200.4s
[resnet50] ep 11/15 │ val_acc=0.8232  f1=0.8196 │ 200.3s
[resnet50] ep 12/15 │ val_acc=0.8324  f1=0.8292 │ 199.7s
[resnet50] ep 13/15 │ val_acc=0.8374  f1=0.8336 │ 200.9s
[resnet50] ep 14/15 │ val_acc=0.8390  f1=0.8359 │ 199.6s
[resnet50] ep 15/15 │ val_acc=0.8434  f1=0.8398 │ 201.0s

[resnet50] TEST: {'acc': 0.8402, 'f1': 0.8402085759461272, 'precision': 0.8420891555087985, 'recall': 0.8402000000000001}


In [7]:
# ──────────────── Training Curves ────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

eps = [h['epoch'] for h in history]
ax1.plot(eps, [h['train_loss'] for h in history], label='Train')
ax1.plot(eps, [h['val_loss']   for h in history], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title(f'{MODEL_NAME} — Loss'); ax1.legend(); ax1.grid(True, alpha=.3)

ax2.plot(eps, [h['train_acc'] for h in history], label='Train')
ax2.plot(eps, [h['val_acc']   for h in history], label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title(f'{MODEL_NAME} — Accuracy'); ax2.legend(); ax2.grid(True, alpha=.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_curves.png', dpi=150,
            bbox_inches='tight')
plt.show()
print('Saved training curves.')

Saved training curves.


In [8]:
# ──────────────── ONNX Export · INT-8 Quantisation · CPU Latency ────────────────

def export_onnx(model, name, size=IMG_SIZE):
    model = model.cpu().eval()
    dummy = torch.randn(1, 3, size, size)
    path  = f'{RESULTS_DIR}/{name}_fp32.onnx'
    torch.onnx.export(model, dummy, path, opset_version=17,
                      input_names=['input'], output_names=['logits'],
                      dynamic_axes={'input': {0:'batch'},
                                    'logits': {0:'batch'}})
    try:
        from onnxsim import simplify
        ms, ok = simplify(onnx.load(path))
        if ok: onnx.save(ms, path)
    except Exception:
        pass
    print(f'Exported  : {path}')
    return path


def quantize_int8(fp32_path):
    int8_path = fp32_path.replace('fp32', 'int8')
    quantize_dynamic(model_input=fp32_path, model_output=int8_path,
                     weight_type=QuantType.QInt8,
                     op_types_to_quantize=['MatMul', 'Gemm'],
                     per_channel=True)
    print(f'Quantised : {int8_path}')
    return int8_path


def bench_cpu(model_path, reps=300, warmup=30, size=IMG_SIZE):
    sess  = ort.InferenceSession(model_path,
                                 providers=['CPUExecutionProvider'])
    iname = sess.get_inputs()[0].name
    x     = np.random.rand(1, 3, size, size).astype('float32')
    for _ in range(warmup):
        sess.run(None, {iname: x})
    times = []
    proc  = psutil.Process(os.getpid())
    mem0  = proc.memory_info().rss
    for _ in range(reps):
        t0 = time.perf_counter()
        sess.run(None, {iname: x})
        times.append((time.perf_counter() - t0) * 1000)
    mem1 = proc.memory_info().rss
    arr  = np.array(times)
    return dict(
        p50_ms      = float(np.percentile(arr, 50)),
        p95_ms      = float(np.percentile(arr, 95)),
        mean_ms     = float(arr.mean()),
        std_ms      = float(arr.std()),
        model_mb    = os.path.getsize(model_path) / 1e6,
        peak_ram_mb = (mem1 - mem0) / 1024 / 1024,
    )


onnx_fp32 = export_onnx(model, f'{MODEL_NAME}_cifar100')
onnx_int8 = quantize_int8(onnx_fp32)

bench_fp32 = bench_cpu(onnx_fp32)
bench_int8 = bench_cpu(onnx_int8)

print(f'\n{MODEL_NAME} FP32 : {bench_fp32}')
print(f'{MODEL_NAME} INT8 : {bench_int8}')
print(f'Speedup (FP32→INT8) : '
      f'{bench_fp32["p50_ms"]/bench_int8["p50_ms"]:.2f}x')

# Move model back to device for subsequent cells
model = model.to(DEVICE)

Exported  : ./results/resnet50_cifar100_fp32.onnx


Quantised : ./results/resnet50_cifar100_int8.onnx

resnet50 FP32 : {'p50_ms': 37.688468499709415, 'p95_ms': 41.61741754960531, 'mean_ms': 38.238677456659694, 'std_ms': 3.1550088277594193, 'model_mb': 94.778338, 'peak_ram_mb': 0.0}
resnet50 INT8 : {'p50_ms': 37.581580999358266, 'p95_ms': 44.22850460000518, 'mean_ms': 38.30876647665415, 'std_ms': 3.724715593675006, 'model_mb': 94.165438, 'peak_ram_mb': 0.0}
Speedup (FP32→INT8) : 1.00x


In [9]:
# ──────────────── ECE · Temperature Scaling · Reliability Diagrams ────────────────

def logits_and_labels(model, loader):
    model.eval()
    Ls, Ys = [], []
    with torch.no_grad():
        for X, Y in loader:
            X = X.to(DEVICE, non_blocking=True)
            Ls.append(model(X).cpu()); Ys.append(Y)
    return torch.cat(Ls), torch.cat(Ys)


def compute_ece(logits, labels, n_bins=15):
    probs = torch.softmax(logits, 1)
    conf, pred = probs.max(1)
    acc  = (pred == labels).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.any():
            ece += (m.float().mean().item()
                    * abs(acc[m].mean().item() - conf[m].mean().item()))
    return float(ece)


class Temperature(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(1))
    def forward(self, z):
        return z / self.log_t.exp()


def fit_temperature(val_logits, val_labels):
    ts  = Temperature()
    nll = nn.CrossEntropyLoss()
    opt = torch.optim.LBFGS(ts.parameters(), lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = nll(ts(val_logits), val_labels)
        loss.backward(); return loss
    opt.step(closure)
    return ts


def reliability_diagram(logits, labels, title, out_path):
    probs = torch.softmax(logits, 1)
    conf, pred = probs.max(1)
    acc = (pred == labels).float()
    bins = torch.linspace(0, 1, 16); xs, ys = [], []
    for i in range(15):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.any():
            xs.append(conf[m].mean().item())
            ys.append(acc[m].mean().item())
    plt.figure(figsize=(6, 6))
    plt.plot([0,1],[0,1],'--', color='gray', label='Perfect')
    plt.bar(xs, ys, width=0.05, alpha=0.7, label='Model')
    plt.xlabel('Confidence'); plt.ylabel('Accuracy')
    plt.title(title); plt.legend(); plt.grid(True, alpha=0.3)
    plt.savefig(out_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f'Saved: {out_path}')


val_logits,  val_labels  = logits_and_labels(model, val_loader)
test_logits, test_labels = logits_and_labels(model, test_loader)

ece_pre  = compute_ece(test_logits, test_labels)
T        = fit_temperature(val_logits, val_labels)
ece_post = compute_ece(T(test_logits).detach(), test_labels)

print(f'\n{MODEL_NAME}  ECE pre-temp : {ece_pre:.4f}')
print(f'{MODEL_NAME}  ECE post-temp: {ece_post:.4f}')

reliability_diagram(
    test_logits, test_labels,
    f'{MODEL_NAME} Pre-Temperature',
    f'{RESULTS_DIR}/{MODEL_NAME}_rel_pre.png')
reliability_diagram(
    T(test_logits).detach(), test_labels,
    f'{MODEL_NAME} Post-Temperature',
    f'{RESULTS_DIR}/{MODEL_NAME}_rel_post.png')


resnet50  ECE pre-temp : 0.0508
resnet50  ECE post-temp: 0.0325
Saved: ./results/resnet50_rel_pre.png
Saved: ./results/resnet50_rel_post.png


In [10]:
# ──────────────── Corruption Robustness ────────────────
import cv2
from PIL import Image

CORRUPTION_NAMES = ['defocus_blur', 'gaussian_noise',
                    'jpeg_compression', 'brightness', 'contrast']

def apply_corruptions(img):
    arr = np.asarray(img)
    blur = cv2.GaussianBlur(arr, (7, 7), 0)
    noise = np.clip(arr/255. + np.random.normal(0, 0.08, arr.shape),
                    0, 1)
    noise = (noise * 255).astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format='JPEG', quality=25)
    buf.seek(0)
    jpeg = np.array(Image.open(buf).convert('RGB'))
    bright = cv2.convertScaleAbs(arr, alpha=1.3, beta=0)
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=1.3, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    contrast = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)
    return [Image.fromarray(x) for x in
            [blur, noise, jpeg, bright, contrast]]


def eval_corrupted(model):
    model.eval()
    base_tf = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    # Clean accuracy
    clean_ok, total = 0, 0
    with torch.no_grad():
        for X, Y in test_loader:
            X, Y = X.to(DEVICE), Y.to(DEVICE)
            clean_ok += (model(X).argmax(1) == Y).sum().item()
            total += Y.size(0)
    clean_acc = clean_ok / total

    # Per-corruption accuracy
    accs = []
    raw_ds = datasets.CIFAR100('./data', train=False, download=False)
    for cidx, cname in enumerate(CORRUPTION_NAMES):
        ok, n = 0, 0
        for img, label in raw_ds:
            c_imgs = apply_corruptions(img)
            x = base_tf(c_imgs[cidx]).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                ok += int(model(x).argmax(1).item() == label)
            n += 1
        accs.append(ok / n)
        print(f'  {cname:20s}: {ok/n:.4f}')

    median_acc = float(np.median(accs))
    drop = clean_acc - median_acc
    return clean_acc, drop, dict(zip(CORRUPTION_NAMES, accs))


print(f'\n{"="*60}')
print(f'  Corruption Robustness — {MODEL_NAME}')
print(f'{"="*60}')
clean_acc, median_drop, corruption_detail = eval_corrupted(model)
print(f'\nClean acc       : {clean_acc:.4f}')
print(f'Median corrupted: '
      f'{float(np.median(list(corruption_detail.values()))):.4f}')
print(f'Accuracy drop   : {median_drop:.4f}')


  Corruption Robustness — resnet50
  defocus_blur        : 0.1216
  gaussian_noise      : 0.0742
  jpeg_compression    : 0.3187
  brightness          : 0.8208
  contrast            : 0.4870

Clean acc       : 0.8402
Median corrupted: 0.3187
Accuracy drop   : 0.5215


In [11]:
# ──────────────── Cross-Domain Generalization  (STL-10 Linear Probe) ────────────────
print(f'\n{"="*60}')
print(f'  Cross-Domain: CIFAR-100 → STL-10  ({MODEL_NAME})')
print(f'{"="*60}')

stl_model = copy.deepcopy(model)
# Freeze the entire backbone
for p in stl_model.parameters():
    p.requires_grad = False

# Replace head with 10-class linear layer and unfreeze it
stl_model.fc = nn.Linear(stl_model.fc.in_features, 10)
for p in stl_model.fc.parameters():
    p.requires_grad = True

stl_model = stl_model.to(DEVICE)
stl_opt = optim.Adam(stl_model.fc.parameters(), lr=1e-3)
stl_crit = nn.CrossEntropyLoss()

print('Training linear probe on STL-10 (5 epochs) ...')
for ep in range(1, 6):
    stl_model.train()
    for X, Y in stl10_train_loader:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        stl_opt.zero_grad()
        loss = stl_crit(stl_model(X), Y)
        loss.backward()
        stl_opt.step()

    # quick eval
    stl_model.eval()
    ok, n = 0, 0
    with torch.no_grad():
        for X, Y in stl10_test_loader:
            X, Y = X.to(DEVICE), Y.to(DEVICE)
            ok += (stl_model(X).argmax(1) == Y).sum().item()
            n  += Y.size(0)
    print(f'  STL-10 ep {ep}: acc={ok/n:.4f}')

# Final STL-10 metrics
stl_model.eval()
stl_true, stl_pred = [], []
with torch.no_grad():
    for X, Y in stl10_test_loader:
        X = X.to(DEVICE)
        stl_true += Y.tolist()
        stl_pred += stl_model(X).argmax(1).cpu().tolist()

stl10_acc = accuracy_score(stl_true, stl_pred)
stl10_f1  = f1_score(stl_true, stl_pred, average='macro')
print(f'\n{MODEL_NAME} STL-10 linear probe  '
      f'acc={stl10_acc:.4f}  f1={stl10_f1:.4f}')


  Cross-Domain: CIFAR-100 → STL-10  (resnet50)
Training linear probe on STL-10 (5 epochs) ...
  STL-10 ep 1: acc=0.7716
  STL-10 ep 2: acc=0.7929
  STL-10 ep 3: acc=0.8061
  STL-10 ep 4: acc=0.8140
  STL-10 ep 5: acc=0.8243

resnet50 STL-10 linear probe  acc=0.8243  f1=0.8228


In [12]:
# ──────────────── Energy Efficiency  (CodeCarbon) ────────────────
print(f'\n{"="*60}')
print(f'  Energy Efficiency — {MODEL_NAME}')
print(f'{"="*60}')

try:
    from codecarbon import EmissionsTracker

    N_INFER = 1000
    sess  = ort.InferenceSession(onnx_int8,
                                 providers=['CPUExecutionProvider'])
    iname = sess.get_inputs()[0].name
    dummy = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype('float32')

    # warm-up
    for _ in range(50):
        sess.run(None, {iname: dummy})

    tracker = EmissionsTracker(
        project_name=f'{MODEL_NAME}_inference',
        measure_power_secs=1,
        log_level='error',
        save_to_file=False,
    )
    tracker.start()

    t_start = time.perf_counter()
    for _ in range(N_INFER):
        sess.run(None, {iname: dummy})
    wall_s = time.perf_counter() - t_start

    emissions_kg = tracker.stop()

    # Extract energy in kWh → Joules
    energy_kwh = (
        tracker._total_energy.kWh
        if hasattr(tracker, '_total_energy') else 0
    )
    energy_joules = energy_kwh * 3_600_000

    energy_results = dict(
        total_joules        = energy_joules,
        joules_per_1k_infer = energy_joules,   # already 1 k
        co2_kg              = float(emissions_kg) if emissions_kg else 0,
        wall_time_s         = wall_s,
        n_inferences        = N_INFER,
    )
    print(f'Energy for {N_INFER} inferences : '
          f'{energy_joules:.4f} J  ({energy_kwh*1e6:.2f} mWh)')
    print(f'CO₂ emissions              : {emissions_kg:.6f} kg')
    print(f'Wall time                  : {wall_s:.2f} s')

except Exception as e:
    print(f'CodeCarbon unavailable or failed: {e}')
    energy_results = dict(total_joules=0, joules_per_1k_infer=0,
                          co2_kg=0, wall_time_s=0, n_inferences=0)


  Energy Efficiency — resnet50


[codecarbon WARNING @ 18:11:15] Multiple instances of codecarbon are allowed to run at the same time.


Energy for 1000 inferences : 2343.1430 J  (650.87 mWh)
CO₂ emissions              : 0.000357 kg
Wall time                  : 37.62 s


In [13]:
# ──────────────── McNemar's Statistical Significance Test ────────────────
print(f'\n{"="*60}')
print(f'  McNemar\'s Test — {MODEL_NAME} vs other models')
print(f'{"="*60}')

# Save this model's predictions for pairwise comparison
model.eval()
all_true, all_pred = [], []
with torch.no_grad():
    for X, Y in test_loader:
        X = X.to(DEVICE)
        all_true += Y.tolist()
        all_pred += model(X).argmax(1).cpu().tolist()

all_true = np.array(all_true)
all_pred = np.array(all_pred)
np.savez(f'{PRED_DIR}/{MODEL_NAME}_preds.npz',
         y_true=all_true, y_pred=all_pred)
print(f'Saved predictions → {PRED_DIR}/{MODEL_NAME}_preds.npz')


def mcnemar_test(y_true, pred_a, pred_b):
    """McNemar's test (with continuity correction).
    Returns (chi2_stat, p_value, b01, b10).
    """
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b01 = int((~correct_a &  correct_b).sum())  # A wrong, B right
    b10 = int(( correct_a & ~correct_b).sum())  # A right, B wrong
    if b01 + b10 == 0:
        return 0.0, 1.0, b01, b10
    stat = (abs(b01 - b10) - 1)**2 / (b01 + b10)
    p    = 1 - chi2.cdf(stat, df=1)
    return float(stat), float(p), b01, b10


# Compare with every other model whose predictions exist
OTHER_MODELS = ['resnet50', 'efficientnet_b0', 'mobilenet_v2', 'vit_b_16']
for other in OTHER_MODELS:
    if other == MODEL_NAME:
        continue
    path = f'{PRED_DIR}/{other}_preds.npz'
    if not os.path.exists(path):
        print(f'  {other:20s}: predictions not found (run that notebook first)')
        continue
    d    = np.load(path)
    stat, p, b01, b10 = mcnemar_test(all_true, all_pred, d['y_pred'])
    sig  = 'YES' if p < 0.05 else 'NO'
    print(f'  {MODEL_NAME} vs {other:20s} │ χ²={stat:8.2f}  '
          f'p={p:.4f}  sig@0.05={sig}  '
          f'(b01={b01}, b10={b10})')


  McNemar's Test — resnet50 vs other models
Saved predictions → ./predictions/resnet50_preds.npz
  efficientnet_b0     : predictions not found (run that notebook first)
  mobilenet_v2        : predictions not found (run that notebook first)
  vit_b_16            : predictions not found (run that notebook first)


In [14]:
# ──────────────── Results Summary ────────────────
print(f'\n{"="*60}')
print(f'  FULL RESULTS — {MODEL_NAME}')
print(f'{"="*60}\n')

summary = {
    'model':              MODEL_NAME,
    'dataset':            'CIFAR-100',
    'num_classes':        NUM_CLASSES,
    'epochs':             EPOCHS,
    'device':             f'GPU ({torch.cuda.get_device_name()})' if DEVICE == 'cuda' else 'CPU',
    'test_metrics':       test_metrics,
    'ece_pre_temp':       ece_pre,
    'ece_post_temp':      ece_post,
    'bench_fp32':         bench_fp32,
    'bench_int8':         bench_int8,
    'clean_acc':          clean_acc,
    'corruption_detail':  corruption_detail,
    'median_acc_drop':    median_drop,
    'stl10_acc':          stl10_acc,
    'stl10_f1':           stl10_f1,
    'energy':             energy_results,
}

# Save JSON
with open(f'{RESULTS_DIR}/{MODEL_NAME}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Pretty print
for k, v in summary.items():
    if isinstance(v, dict):
        print(f'{k}:')
        for kk, vv in v.items():
            print(f'    {kk:25s}: {vv}')
    else:
        print(f'{k:25s}: {v}')

print(f'\nJSON saved → {RESULTS_DIR}/{MODEL_NAME}_summary.json')
print('\n✓ Notebook complete.')


  FULL RESULTS — resnet50

model                    : resnet50
dataset                  : CIFAR-100
num_classes              : 100
epochs                   : 15
device                   : GPU (Tesla P100-PCIE-16GB)
test_metrics:
    acc                      : 0.8402
    f1                       : 0.8402085759461272
    precision                : 0.8420891555087985
    recall                   : 0.8402000000000001
ece_pre_temp             : 0.05084816558829963
ece_post_temp            : 0.032491985699682834
bench_fp32:
    p50_ms                   : 37.688468499709415
    p95_ms                   : 41.61741754960531
    mean_ms                  : 38.238677456659694
    std_ms                   : 3.1550088277594193
    model_mb                 : 94.778338
    peak_ram_mb              : 0.0
bench_int8:
    p50_ms                   : 37.581580999358266
    p95_ms                   : 44.22850460000518
    mean_ms                  : 38.30876647665415
    std_ms                   : 3.7247155